In [1]:
# Run this once to clear the cache
import shutil, os

cache = '/content/drive/MyDrive/dataset/oasis13d/src/__pycache__'
if os.path.exists(cache):
    shutil.rmtree(cache)
    print("✅ cache cleared")

# Then restart the Colab runtime:
# Runtime menu → Restart Runtime (Ctrl+M .)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/dataset/oasis13d')

Mounted at /content/drive


In [32]:
# Double check Python actually sees the path
import sys
print([p for p in sys.path if 'oasis' in p])
# Must print: ['/content/drive/MyDrive/dataset/oasis13d']

['/content/drive/MyDrive/dataset/oasis13d', '/content/drive/MyDrive/dataset/oasis13d', '/content/drive/MyDrive/dataset/oasis13d', '/content/drive/MyDrive/dataset/oasis13d', '/content/drive/MyDrive/dataset/oasis13d', '/content/drive/MyDrive/dataset/oasis13d/data', '/content/drive/MyDrive/dataset/oasis13d/data', '/content/drive/MyDrive/dataset/oasis13d', '/content/drive/MyDrive/dataset/oasis13d/data', '/content/drive/MyDrive/dataset/oasis13d', '/content/drive/MyDrive/dataset/oasis13d/data', '/content/drive/MyDrive/dataset/oasis13d/data', '/content/drive/MyDrive/dataset/oasis13d', '/content/drive/MyDrive/dataset/oasis13d', '/content/drive/MyDrive/dataset/oasis13d/data', '/content/drive/MyDrive/dataset/oasis13d']


In [3]:
from src.dataset import build_subject_list, OASISDataset
print("✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅


In [4]:
from src.preprocessing import (
    ClinicalPreprocessor, compute_class_weights,
    N_FEATURES, N_CLASSES, FEATURE_COLS, CDR_TO_CLASS
)
print("✅ preprocessing imported")

✅ preprocessing imported


In [5]:
from src.encoder_clinical import ClinicalMLPEncoder
import torch

encoder = ClinicalMLPEncoder()
dummy = torch.randn(4, 7)
out = encoder(dummy)
print(f"✅ encoder_clinical works — output: {tuple(out.shape)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ encoder_clinical works — output: (4, 1, 256)


In [6]:
from src.encoder_mri import BrainMRI3DEncoder
import torch

encoder = BrainMRI3DEncoder()
dummy = torch.randn(2, 1, 96, 96, 96)
patches, global_vec = encoder(dummy)
print(f"✅ encoder_mri works — patches: {tuple(patches.shape)}, global: {tuple(global_vec.shape)}")

✅ encoder_mri works — patches: (2, 216, 256), global: (2, 256)


In [10]:
import os

mri_root = '/content/drive/MyDrive/dataset/oasis13d/data'

for root, dirs, files in os.walk(mri_root):
    # Only show first 3 subjects to avoid flooding output
    nii_files = [f for f in files if f.endswith('.nii') or f.endswith('.nii.gz')]
    if nii_files:
        print(root)
        for f in nii_files[:2]:
            print(f'    {f}')
        break  # remove this break to see all

# Also print top-level folders
print('\nTop-level folders:')
for item in sorted(os.listdir(mri_root))[:10]:
    print(f'  {item}')

/content/drive/MyDrive/dataset/oasis13d/data/oasis/OASIS
    OAS1_0001_MR1_mpr_n4_anon_sbj_111_normalised.nii
    OAS1_0003_MR1_mpr_n4_anon_sbj_111_normalised.nii

Top-level folders:
  oasis
  oasis_cross-sectional.csv
  oasis_cross-sectional_facts.pdf


In [26]:
import os

mri_root = '/content/drive/MyDrive/dataset/oasis13d'

all_files = sorted(os.listdir(mri_root))

# Show first 10 files
print("First 10 files:")
for f in all_files[:10]:
    print(f'  {f}')

print(f"\nTotal files: {len(all_files)}")

First 10 files:
  .vscode
  archive (1).zip
  data
  notebooks
  src
  structure.txt
  train.py

Total files: 7


In [7]:
!pip install nibabel -q

from src.dataset import build_subject_list, OASISDataset
from src.preprocessing import ClinicalPreprocessor, CDR_TO_CLASS
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

BASE    = '/content/drive/MyDrive/dataset/oasis13d/data'
CSV     = f'{BASE}/oasis_cross-sectional.csv'
NII_DIR = f'{BASE}/oasis/OASIS'

df     = build_subject_list(CSV, NII_DIR)
labels = df['CDR'].map(CDR_TO_CLASS).values
df_tr, df_va = train_test_split(df, test_size=0.2, random_state=42, stratify=labels)

prep = ClinicalPreprocessor()
prep.fit_transform(df_tr)

train_ds = OASISDataset(df_tr, prep, augment=True)
loader   = DataLoader(train_ds, batch_size=2, shuffle=True)

mri, clinical, label = next(iter(loader))
print(f"MRI      : {tuple(mri.shape)}")
print(f"Clinical : {tuple(clinical.shape)}")
print(f"Label    : {label.tolist()}")

NIfTI scanner   : found 416 subjects in /content/drive/MyDrive/dataset/oasis13d/data/oasis/OASIS
Subject list    : 416 subjects with both CSV and NIfTI
Feature matrix : (332, 7)  (N_samples × 7 features)
Features       : ['Age', 'M/F', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV']
Value range    : [-4.84, 3.22]  (after StandardScaler)
Class distribution:
  class 0 (Non-demented): ████████████████████████████████████████████████████████████ 252
  class 1 (Very mild): ████████████████████████████████████████████████████████ 56
  class 2 (Mild/Moderate): ████████████████████████ 24
OASISDataset    : 332 subjects  | augment=True  | shape=(96, 96, 96)
MRI      : (2, 1, 96, 96, 96)
Clinical : (2, 7)
Label    : [1, 1]


In [8]:
from src.fusion import CrossModalFusion, attention_to_volume
import torch

D, B = 256, 4
patches    = torch.randn(B, 216, D)
global_vec = torch.randn(B, D)
clin_token = torch.randn(B, 1, D)

fusion = CrossModalFusion()
fused, attn_w = fusion(patches, global_vec, clin_token)

print(f"fused    : {tuple(fused.shape)}")     # (4, 256)
print(f"attn_w   : {tuple(attn_w.shape)}")    # (4, 8, 1, 216)
print("✅ fusion works")

fused    : (4, 256)
attn_w   : (4, 4, 1, 216)
✅ fusion works


In [1]:
import os
import shutil

LOCAL_CACHE = '/content/mri_cache'
NII_DIR     = '/content/drive/MyDrive/dataset/oasis13d/data/oasis/OASIS'

os.makedirs(LOCAL_CACHE, exist_ok=True)

files = [f for f in os.listdir(NII_DIR) if f.endswith('.nii')]
print(f"Copying {len(files)} files to local cache...")

for i, f in enumerate(files):
    src = os.path.join(NII_DIR, f)
    dst = os.path.join(LOCAL_CACHE, f)
    
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
    
    if (i+1) % 50 == 0:
        print(f"{i+1}/{len(files)} copied")

print("✅ All files cached locally")

Copying 436 files to local cache...
50/436 copied
100/436 copied
150/436 copied
200/436 copied
250/436 copied
300/436 copied
350/436 copied
400/436 copied
✅ All files cached locally


In [9]:
from src.model import AlzheimerFusionModel, model_summary
import torch

B = 2
dummy_mri  = torch.randn(B, 1, 96, 96, 96)
dummy_clin = torch.randn(B, 7)

# Test all modes
for mode in AlzheimerFusionModel.VALID_MODES:
    m = AlzheimerFusionModel(mode=mode)
    m.eval()
    with torch.no_grad():
        logits, attn_w = m(dummy_mri, dummy_clin)
    attn_str = str(tuple(attn_w.shape)) if attn_w is not None else 'None'
    print(f"{mode:<16} logits={tuple(logits.shape)}  attn={attn_str}")

# Full model summary
print()
model_summary(AlzheimerFusionModel(mode='full'))

full             logits=(2, 3)  attn=(2, 4, 1, 216)
mri_only         logits=(2, 3)  attn=None
clinical_only    logits=(2, 3)  attn=None
concat_only      logits=(2, 3)  attn=None


Component                        Parameters     Trainable
──────────────────────────────────────────────────────────
  mri_encoder                     4,586,976     4,586,976
  clin_encoder                       42,808        42,808
  fusion                            461,313       461,313
  classifier                         33,539        33,539
──────────────────────────────────────────────────────────
  TOTAL                           5,124,636     5,124,636



In [ ]:
!pip install nibabel -q

import sys
sys.path.insert(0, '/content/drive/MyDrive/dataset/oasis13d')

%run /content/drive/MyDrive/dataset/oasis13d/train.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device : cuda
GPU    : Tesla T4

── Loading data ──
NIfTI scanner   : found 416 subjects in /content/drive/MyDrive/dataset/oasis13d/data/oasis/OASIS
Subject list    : 416 subjects with both CSV and NIfTI
Train: 332  Val: 84
Feature matrix : (332, 7)  (N_samples × 7 features)
Features       : ['Age', 'M/F', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV']
Value range    : [-4.84, 3.22]  (after StandardScaler)
Class distribution:
  class 0 (Non-demented): ████████████████████████████████████████████████████████████ 252
  class 1 (Very mild): ████████████████████████████████████████████████████████ 56
  class 2 (Mild/M

In [33]:
import os

mri_root = '/content/drive/MyDrive/dataset/oasis13d'
train_file = os.path.join(mri_root, 'train.py')

# Check if file exists
if os.path.exists(train_file):
    print(f"train.py found at: {train_file}")
    
    # Read first few lines to see content
    with open(train_file, 'r') as f:
        content = f.read()
        print("\nFirst 500 characters of train.py:")
        print(content[:500])
else:
    print(f"train.py not found at {train_file}")
    print(f"Files in {mri_root}:")
    for f in os.listdir(mri_root):
        print(f"  {f}")

train.py found at: /content/drive/MyDrive/dataset/oasis13d/train.py

First 500 characters of train.py:
"""
train.py
--------
Training script for AlzheimerFusionModel.

Staged training strategy (for 332 training subjects):
  Stage 1 — freeze MRI encoder, train fusion + classifier (~30 epochs)
             Goal: learn meaningful fusion without overfitting the 3.5M-param CNN
  Stage 2 — unfreeze MRI encoder, fine-tune everything with differential lr (~20 epochs)
             Goal: fine-tune MRI features now that fusion is stable

Usage (in Colab):
    !python train.py

Or import and call directly in
